## 2. Entendimento dos dados

Os valores ausentes estão codificados como **`-200`**. O primeiro passo é convertê-los para `NaN` e medir a ausência por variável **antes de escolher qualquer imputação**.

In [ ]:
with DATA_PATH.open('r', encoding='utf-8', newline='') as f:
    rows = list(csv.reader(f))

headers = rows[0]
raw = rows[1:]

excel_epoch = datetime(1899, 12, 30)
datetimes = np.array([
    excel_epoch + timedelta(days=float(r[0]) + float(r[1]))
    for r in raw
], dtype=object)

feature_names_all = np.array(headers[2:], dtype=object)
X_raw = np.array([[float(v) for v in r[2:]] for r in raw], dtype=float)
X_raw[X_raw == -200] = np.nan

print(f'Observações: {X_raw.shape[0]:,}')
print(f'Variáveis analíticas: {X_raw.shape[1]}')
print('Período:', datetimes.min(), 'até', datetimes.max())

In [ ]:
missing_rate_all = np.isnan(X_raw).mean(axis=0)
order = np.argsort(missing_rate_all)[::-1]

html = ['<table><tr><th>Variável</th><th>Ausentes</th><th>% ausente</th></tr>']
for j in order:
    html.append(
        f'<tr><td>{feature_names_all[j]}</td>'
        f'<td>{int(np.isnan(X_raw[:, j]).sum()):,}</td>'
        f'<td>{100*missing_rate_all[j]:.2f}%</td></tr>'
    )
html.append('</table>')
display(HTML(''.join(html)))

plt.figure(figsize=(10, 4))
plt.bar(feature_names_all, 100 * missing_rate_all)
plt.axhline(50, linestyle='--')
plt.ylabel('% de valores ausentes')
plt.title('Ausência por variável após converter -200 para NaN')
plt.xticks(rotation=75, ha='right')
plt.tight_layout()
plt.show()

### Regra de qualidade adotada

- variáveis com **mais de 50%** de ausência são retiradas do espaço de modelagem, pois sua imputação passaria a dominar a informação real;
- observações com **mais de 30%** das variáveis ausentes serão marcadas como **baixa qualidade** e não poderão gerar alerta de anomalia;
- para as demais lacunas, a estratégia de imputação será **avaliada empiricamente**, simulando blocos ausentes de 1 a 24 horas.

Essa separação é importante: **dado ruim não é anomalia operacional**.

In [ ]:
KEEP_THRESHOLD = 0.50
keep_columns = missing_rate_all <= KEEP_THRESHOLD

dropped_features = feature_names_all[~keep_columns].tolist()
feature_names = feature_names_all[keep_columns]
X = X_raw[:, keep_columns]
row_missing_rate = np.isnan(X).mean(axis=1)
low_quality = row_missing_rate > 0.30

print('Variáveis mantidas:', len(feature_names))
print('Variáveis descartadas:', dropped_features)
print(f'Linhas de baixa qualidade (>30% ausente): {low_quality.sum():,} ({100*low_quality.mean():.2f}%)')